# liel × ソーシャルネットワーク分析

liel をサーバーレスのグラフDBとして使い、ソーシャルグラフを構築・分析します。

**カバー内容**
- グラフの構築と保存
- 隣接・BFS・最短経路クエリ
- pandas DataFrame への変換と集計
- networkx + matplotlib で可視化
- JSON / CSV エクスポート

```
pip install liel pandas networkx matplotlib
```

In [ ]:
import liel
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"liel version: {liel.__version__}")

## 1. グラフを構築する

6人の「Person」ノードと FOLLOWS エッジでソーシャルグラフを作ります。  
`:memory:` を指定するとインメモリ動作（ファイル保存不要）。

In [ ]:
db = liel.open(":memory:")

people = [
    ("Alice", 30, "Engineer"),
    ("Bob",   25, "Designer"),
    ("Carol", 35, "Manager"),
    ("Dave",  28, "Engineer"),
    ("Eve",   32, "Designer"),
    ("Frank", 40, "Manager"),
]
nodes = {}
for name, age, role in people:
    nodes[name] = db.add_node(["Person"], name=name, age=age, role=role)

follows = [
    ("Alice", "Bob",   2020),
    ("Alice", "Carol", 2019),
    ("Bob",   "Dave",  2021),
    ("Carol", "Dave",  2022),
    ("Carol", "Eve",   2020),
    ("Dave",  "Frank", 2023),
    ("Eve",   "Frank", 2021),
    ("Frank", "Alice", 2018),
]
for frm, to, since in follows:
    db.add_edge(nodes[frm], "FOLLOWS", nodes[to], since=since)

db.commit()
print(f"{db.node_count()} nodes, {db.edge_count()} edges")

## 2. 基本クエリ

In [ ]:
# Alice がフォローしている人
following = db.neighbors(nodes["Alice"], edge_label="FOLLOWS", direction="out")
print("Alice → follows:", [n["name"] for n in following])

# Alice をフォローしている人
followers = db.neighbors(nodes["Alice"], edge_label="FOLLOWS", direction="in")
print("Alice ← followers:", [n["name"] for n in followers])

In [ ]:
# BFS: Alice から 2 ホップ以内の全員
reachable = db.bfs(nodes["Alice"], max_depth=2)
for node, depth in reachable:
    print(f"  depth={depth}  {node['name']}")

In [ ]:
# Alice → Frank の最短経路
path = db.shortest_path(nodes["Alice"], nodes["Frank"], edge_label="FOLLOWS")
if path:
    print(" → ".join(n["name"] for n in path))
else:
    print("経路なし")

In [ ]:
# QueryBuilder: 30歳以上の Engineer
senior_engineers = (
    db.nodes()
      .label("Person")
      .where_(lambda n: n.get("age") >= 30 and n.get("role") == "Engineer")
      .fetch()
)
for n in senior_engineers:
    print(f"  {n['name']}  age={n['age']}")

## 3. pandas DataFrame 変換

In [ ]:
def nodes_to_df(db):
    rows = []
    for node in db.all_nodes():
        row = {"id": node.id, "labels": node.labels}
        row.update(node.properties)
        rows.append(row)
    return pd.DataFrame(rows)

def edges_to_df(db):
    rows = []
    for edge in db.all_edges():
        row = {"id": edge.id, "label": edge.label,
               "from_node": edge.from_node, "to_node": edge.to_node}
        row.update(edge.properties)
        rows.append(row)
    return pd.DataFrame(rows)

node_df = nodes_to_df(db)
node_df

In [ ]:
edge_df = edges_to_df(db)
edge_df

In [ ]:
# 役職別 平均年齢
node_df.groupby("role")["age"].agg(["mean", "count"])

In [ ]:
# フォロワー数ランキング (in-degree)
in_deg = edge_df.groupby("to_node").size().rename("followers")
node_df.set_index("id").join(in_deg).fillna({"followers": 0}).sort_values("followers", ascending=False)

## 4. 可視化 (networkx + matplotlib)

In [ ]:
def to_networkx(db, directed=True):
    G = nx.DiGraph() if directed else nx.Graph()
    for node in db.all_nodes():
        G.add_node(node.id, **node.properties)
    for edge in db.all_edges():
        G.add_edge(edge.from_node, edge.to_node, label=edge.label, **edge.properties)
    return G

G = to_networkx(db)
pos = nx.spring_layout(G, seed=42)

roles = list({G.nodes[n]["role"] for n in G.nodes()})
palette = {r: plt.cm.Set2(i / len(roles)) for i, r in enumerate(roles)}
colors = [palette[G.nodes[n]["role"]] for n in G.nodes()]
labels = {n: G.nodes[n]["name"] for n in G.nodes()}

fig, ax = plt.subplots(figsize=(10, 7))
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=900, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=18, ax=ax,
                       edge_color="#888", connectionstyle="arc3,rad=0.08")
nx.draw_networkx_edge_labels(G, pos,
    edge_labels={(u,v): G.edges[u,v]["since"] for u,v in G.edges()},
    font_size=7, ax=ax)
patches = [mpatches.Patch(color=palette[r], label=r) for r in roles]
ax.legend(handles=patches, loc="upper left")
ax.set_title("ソーシャルグラフ (role 別カラー)")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# PageRank で影響力を数値化
# NetworkX の pagerank は環境によって SciPy を要求するため、
# notebook では小さな純 Python 実装を使います。
def pagerank_simple(G, alpha=0.85, iterations=50):
    nodes = list(G.nodes())
    score = {n: 1 / len(nodes) for n in nodes}
    base = (1 - alpha) / len(nodes)
    for _ in range(iterations):
        next_score = {n: base for n in nodes}
        for src in nodes:
            out = list(G.successors(src))
            if not out:
                share = alpha * score[src] / len(nodes)
                for dst in nodes:
                    next_score[dst] += share
            else:
                share = alpha * score[src] / len(out)
                for dst in out:
                    next_score[dst] += share
        score = next_score
    return score

pr = pagerank_simple(G)
names = {n.id: n["name"] for n in db.all_nodes()}

pr_df = pd.DataFrame([
    {"name": names[nid], "pagerank": score}
    for nid, score in pr.items()
]).sort_values("pagerank", ascending=False).reset_index(drop=True)
pr_df

## 5. JSON / CSV エクスポート

In [ ]:
import csv, json, pathlib

def export_json(db, path):
    data = {
        "nodes": [{"id": n.id, "labels": n.labels, "properties": n.properties}
                  for n in db.all_nodes()],
        "edges": [{"id": e.id, "label": e.label,
                   "from_node": e.from_node, "to_node": e.to_node,
                   "properties": e.properties}
                  for e in db.all_edges()],
    }
    pathlib.Path(path).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Exported {len(data['nodes'])} nodes, {len(data['edges'])} edges -> {path}")

def export_nodes_csv(db, path):
    nodes = db.all_nodes()
    prop_keys = sorted({k for n in nodes for k in n.keys()})
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["id", "labels"] + prop_keys)
        w.writeheader()
        for node in nodes:
            row = {"id": node.id, "labels": "|".join(node.labels)}
            for k in prop_keys:
                row[k] = node.get(k)
            w.writerow(row)
    print(f"Exported {len(nodes)} nodes -> {path}")

base_dir = pathlib.Path.cwd()
data_dir = base_dir / "notebooks" / "data" if (base_dir / "notebooks").exists() else base_dir / "data"
export_dir = data_dir / "social_network"
export_dir.mkdir(parents=True, exist_ok=True)
json_path = export_dir / "social_graph.json"
nodes_csv_path = export_dir / "nodes.csv"

export_json(db, json_path)
export_nodes_csv(db, nodes_csv_path)
print("export dir:", export_dir)

In [ ]:
# エクスポートした CSV を pandas で読み込む
pd.read_csv(nodes_csv_path)

In [ ]:
# JSON 内容を確認
data = json.loads(json_path.read_text(encoding="utf-8"))
print(f"nodes: {len(data['nodes'])}, edges: {len(data['edges'])}")
data["nodes"][0]

## 6. ファイルへの永続化

`:memory:` をファイルパスに変えるだけで永続化できます。

In [ ]:
import uuid

path = data_dir / f"people_demo_{uuid.uuid4().hex}.liel"

db2 = liel.open(str(path))
db2.add_node(["Person"], name="Taro", age=22)
db2.add_node(["Person"], name="Hanako", age=27)
db2.commit()
db2.close()

# 再オープン
db3 = liel.open(str(path))
print(f"再オープン後: {db3.node_count()} nodes")
for n in db3.all_nodes():
    print(f"  {n['name']}  age={n['age']}")
db3.close()
print("saved file:", path)